## Data Engineer Assessment for ABInBev

In [0]:
#Initialization
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window



spark = SparkSession.builder.appName("AbInbev").config("spark.memory.offHeap.enabled","true").config("spark.memory.offHeap.size","10g").getOrCreate()

In [0]:
# File path in DBFS
locations_path = "/Volumes/workspace/default/covid/locations.csv"
vaccinations_path = "/Volumes/workspace/default/covid/vaccinations.json"

In [0]:
# Bronze layer: Reading File 
# (*** Simulating Bronze layer as DataFrame, just for the exercise purpose, in real world would create the medallion database)
df_locations_bronze = spark.read.option("header", True).csv("/Volumes/workspace/default/covid/locations.csv")
df_vaccinations_bronze = spark.read.option("multiLine", True).json("/Volumes/workspace/default/covid/vaccinations.json")

In [0]:
# Initial Data Analysis
df_locations_bronze.show(5)
df_vaccinations_bronze.show(10)

+-----------+--------+--------------------+---------------------+--------------------+--------------------+
|   location|iso_code|            vaccines|last_observation_date|         source_name|      source_website|
+-----------+--------+--------------------+---------------------+--------------------+--------------------+
|Afghanistan|     AFG|CanSino, Covaxin,...|           2023-12-31|World Health Orga...|https://data.who....|
|    Albania|     ALB|Oxford/AstraZenec...|           2023-09-10|World Health Orga...|https://data.who....|
|    Algeria|     DZA|Oxford/AstraZenec...|           2022-09-04|World Health Orga...|https://data.who....|
|    Andorra|     AND|Moderna, Oxford/A...|           2023-09-24|World Health Orga...|https://data.who....|
|     Angola|     AGO|  Oxford/AstraZeneca|           2023-12-31|World Health Orga...|https://data.who....|
+-----------+--------+--------------------+---------------------+--------------------+--------------------+
only showing top 5 rows
+---

## Exercise 2
### What country(s) use more kind of vaccines?

In [0]:

# Conta quantos tipos de vacina por país
df_locations_counts = df_locations_bronze.withColumn("num_vaccines", size(split(col("vaccines"), ", ")))

#df_locations_counts.show(10)

# Ordena para mostrar o país com mais tipos diferentes

print("Country with more kind of vaccines:")
df_locations_counts.orderBy(desc("num_vaccines")).select("location", "num_vaccines", "vaccines").show(1)


Country with more kind of vaccines:
+--------+------------+--------------------+
|location|num_vaccines|            vaccines|
+--------+------------+--------------------+
|    Iran|          12|COVIran Barekat, ...|
+--------+------------+--------------------+
only showing top 1 row


## Exercise 2

###Top 10 country that had more vaccinations per month and year (

In [0]:
# Silver Layer
# Removing regional Totals (iso_code starts with 'OWID_')


# Eplode array data
df_vaccinations_array = df_vaccinations_bronze.withColumn("record", explode("data"))

# Structure the data in the necessary format and filter total regions
df_vaccinations_silver = df_vaccinations_array.select(
    col("country").alias("location"),
    col("iso_code"),
    col("record.date").alias("date"),
    col("record.daily_vaccinations").alias("daily_vaccinations")
).filter(~col("iso_code").like("OWID_%")) \
 .withColumn("date", to_date("date", "yyyy-MM-dd")) \
 .withColumn("year", year("date")) \
 .withColumn("month", month("date"))

df_vaccinations_silver.show(5)

+-----------+--------+----------+------------------+----+-----+
|   location|iso_code|      date|daily_vaccinations|year|month|
+-----------+--------+----------+------------------+----+-----+
|Afghanistan|     AFG|2021-02-22|              NULL|2021|    2|
|Afghanistan|     AFG|2021-02-23|              1367|2021|    2|
|Afghanistan|     AFG|2021-02-24|              1367|2021|    2|
|Afghanistan|     AFG|2021-02-25|              1367|2021|    2|
|Afghanistan|     AFG|2021-02-26|              1367|2021|    2|
+-----------+--------+----------+------------------+----+-----+
only showing top 5 rows


In [0]:
#GOLD LAYER

# Aggregation: Vaccines per yer month
df_vaccinations_gold = df_vaccinations_silver.groupBy("location", "year", "month").agg(
    sum("daily_vaccinations").alias("total_vaccinations")
)

#  Top 10 by Year/Month
df_vaccinated_top10_year_month = df_vaccinations_gold \
    .withColumn("rank", row_number().over(
        Window.partitionBy("year", "month").orderBy(col("total_vaccinations").desc())
    )) \
    .filter(col("rank") <= 10) \
    .select("year", "month", "rank", "location", "total_vaccinations") \
    .orderBy("year", "month", "rank")


print("Top 10 by Year/Month")
display(df_vaccinated_top10_year_month)



Top 10 by Year/Month


year,month,rank,location,total_vaccinations
2020,12,1,United States,4171482
2020,12,2,China,3000000
2020,12,3,Israel,630841
2020,12,4,Russia,441571
2020,12,5,Germany,142021
2020,12,6,Canada,70754
2020,12,7,Poland,36650
2020,12,8,Bahrain,34657
2020,12,9,Argentina,31556
2020,12,10,Hungary,18606


## Exercise 2
### Top 10 country vaccinations per year, all the vaccine used during the fight against covid, ordering the top 10, first by most vaccine used and most vaccinated in the year.


In [0]:
# Aggregation: Vaccines per year

df_vaccinated_year = df_vaccinations_gold.groupBy("location", "year").agg(
    sum("total_vaccinations").alias("total_vaccinations")
)

# Join Locations and Vaccines data
df_vaccinated_location = df_vaccinated_year.join(df_locations_counts, on="location", how="inner")


# Result: top 10 by Year/Month
df_vacc_loc_top10_year = df_vaccinated_location \
    .withColumn("rank", row_number().over(
        Window.partitionBy("year").orderBy(col("num_vaccines").desc(),
                                           col("total_vaccinations").desc()
                                           )
    )) \
    .filter(col("rank") <= 10) \
    .select("year", "rank", "location",  "num_vaccines", "total_vaccinations", "vaccines") \
    .orderBy("year", "rank")


print("Top 10 by Year and by most vaccine used and most vaccinated in the year.")
display(df_vacc_loc_top10_year)



Top 10 by Year and by most vaccine used and most vaccinated in the year.


year,rank,location,num_vaccines,total_vaccinations,vaccines
2020,1,Bahrain,10,34657,"CanSino, Covaxin, Johnson&Johnson, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing, Sinovac, Sputnik Light, Sputnik V"
2020,2,Qatar,10,12159,"CanSino, Covaxin, Johnson&Johnson, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing, Sinovac, Sputnik Light, Sputnik V"
2020,3,Oman,10,4909,"CanSino, Covaxin, Johnson&Johnson, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing, Sinovac, Sputnik Light, Sputnik V"
2020,4,Kuwait,10,3363,"CanSino, Covaxin, Johnson&Johnson, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing, Sinovac, Sputnik Light, Sputnik V"
2020,5,China,7,3000000,"CanSino, IMBCAMS, KCONVAC, Sinopharm/Beijing, Sinopharm/Wuhan, Sinovac, ZF2001"
2020,6,Mexico,7,16059,"CanSino, Johnson&Johnson, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, Sinovac, Sputnik V"
2020,7,Germany,6,142021,"Johnson&Johnson, Moderna, Novavax, Oxford/AstraZeneca, Pfizer/BioNTech, Valneva"
2020,8,Canada,6,70754,"Johnson&Johnson, Medicago, Moderna, Novavax, Oxford/AstraZeneca, Pfizer/BioNTech"
2020,9,Argentina,6,31556,"CanSino, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing, Sputnik V"
2020,10,Hungary,6,18606,"Johnson&Johnson, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing, Sputnik V"
